In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# Now go to the project folder

%cd gdrive/My Drive/

In [ ]:
# Chek the files in the folder to confirm

!ls

agecache.txt	gendercache.txt				   predict.py	 Test_food
app.py		glucosecache.txt			   __pycache__	 User.txt
bloodcache.txt	labels.txt				   ramen.jpg	 utils.py
donuts.jpg	lasagna.jpg				   Run.ipynb	 waffles.jpg
engine.py	Nutrition_Dataset_updated.csv		   static	 yolov8s-seg-v1.onnx
food_info.py	Nutrition_Dataset_updated_with_labels.csv  sushi.jpg
Food_model	omlet.jpg				   takoyaki.jpg
food.txt	pizza.jpg				   templates


In [ ]:
!pip install pyngrok

In [ ]:
!pip install onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 25.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.4 MB/s eta 0:00:00


In [ ]:
!pip install python-dotenv

In [ ]:
from pyngrok import ngrok
from flask import Flask, request, render_template,jsonify # Import flask libraries
# Import the python file containing the ML model
from flask import Flask, request, render_template,jsonify # Import flask libraries
import pickle
import pandas as pd
import string
import cv2
import numpy as np

from keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.models import load_model
from keras.applications.inception_v3 import InceptionV3
from keras.layers import AveragePooling2D, Dropout, Dense, Flatten
from keras.models import Model
import math


import typing
import glob
from PIL import Image
from io import BytesIO
import json

from predict import predict
from food_info import fds_food_info
import matplotlib.pyplot as plt


In [ ]:
port_no = 5000

In [ ]:
#Initialize the flask App
app = Flask(__name__)
#ngrok.set_auth_token("2dYHJdfEwsgk7PE47M4SEHPH6Bg_2rdknf4YCZGU2Sxg3eYtg")
ngrok.set_auth_token("2e1SiKOEZRpK9Mb9jGjRQUiWr5S_3VbU6KwxqYQikpAdXgA7h")
public_url =  ngrok.connect(port_no).public_url



# Id the decision is not to eat the food
# The system will send mail to teh user

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

message = MIMEMultipart()
message["To"] = 'To line here.'
message["From"] = 'The Diabetes Management Portal'
message["Subject"] = 'Your Diabetes Wellness Alert'

title = '<b> Title line here. </b>'
messageText = MIMEText('Dear Sir/Madam, Your recent glucose levels indicate concern, please prioritize your diet and management for optimal health. Contact our Phicision at 1800 800 4025','html')
message.attach(messageText)

email = "shamsudheenmarakkar17@gmail.com"
password =  "ojduepawovrjnvph"




# Default route set as 'home'
@app.route('/')
def landing():
    return render_template('1.landing.html') # Render home.html



@app.route('/login',methods=['POST'])
def login():
	if request.method == 'POST':
		return render_template('2.login.html') # Render home.html


@app.route('/status',methods=['POST'])
def logincheck():
	if request.method == 'POST':
		details = [x for x in request.form.values()]
		print(details)
	username = details[0]
	password = details[1]
	age = details[2]
	gender = details[3]
	glucose_level = details[4]
	blood_group = details[5]

	file = open('agecache.txt', 'w')
	file.write(age)
	file.close()

	file = open('gendercache.txt', 'w')
	file.write(gender)
	file.close()

	file = open('glucosecache.txt', 'w')
	file.write(glucose_level)
	file.close()

	file = open('bloodcache.txt', 'w')
	file.write(blood_group)
	file.close()

	with open('User.txt') as file:
		lines = [line.rstrip() for line in file]
		print('lines are', lines)

	if username.strip() ==lines[0].strip()  and password.strip() ==lines[1].strip() :
		print('match')
		template = '4.image.html'
	elif username!=lines[0].strip() or password != lines[1].strip() :
		print('No')
		template = '3.loginfail.html'


	return render_template(template)



@app.route('/image',methods=['POST'])
def image():
	if request.method == 'POST':
		f=request.form['csvfile']
		if not f:
			print('Error')
	print(f)
	image_file  = f
	img = cv2.imread(f)
	print(img.shape)

	n_classes = 101

	# base model is inception_v3 weights pre-trained on ImageNet
	base_model = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(299,299,3)
		)

	x = base_model.output

	# added layers to the base model
	x = AveragePooling2D(pool_size=(8, 8))(x)
	x = Dropout(.4)(x)
	x = Flatten()(x)

	# add softmax activation
	predictions = Dense(n_classes, activation='softmax')(x)

	model = Model(inputs=base_model.input, outputs=predictions)

	## Load the class labels
	with open('labels.txt', 'r') as f:
		food101 = [l.strip().lower() for l in f]
	model.load_weights('Food_model/food101.h5')
	img = cv2.resize(img, (299, 299))
	print(img.shape)
	img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
	img = np.expand_dims(img, axis=0)
	img = preprocess_input(img)
	predicted_vec = model.predict(img)
	predicted_label = food101[np.argmax(predicted_vec)]
	print(predicted_label)


	food = predicted_label
	food = food.capitalize()
	# save the food detected
	file = open('food.txt', 'w')
	file.write(food)
	file.close()

	#food = 'Omelette'



	model_path = 'yolov8s-seg-v1.onnx'
	image_array = plt.imread(image_file)
	print(image_array.shape, ' Got')
	results = predict(image_array, 'yolov8s-seg-v1.onnx')
	# counting the number of pixels
	out = results['masks'][0]
	number_of_Food_pix = np.sum(out == 1)
	number_of_BG_pix = np.sum(out == 0)

	print('Number of white/Food pixels:', number_of_Food_pix)
	print('Number of black/Background pixels:', number_of_BG_pix)

	number_of_Food_pix
	number_of_BG_pix
	total_pix = number_of_Food_pix + number_of_BG_pix
	total_pix

	percentage_Food_pix = (number_of_Food_pix/total_pix)*100
	if percentage_Food_pix > 50:
		Quantity = number_of_Food_pix/5000
	elif percentage_Food_pix < 50:
		Quantity = number_of_Food_pix/100

	serving = Quantity


	print('Serving quantity is ',Quantity, ' gms')

	#serving = 500.0


	dataNutrition = pd.read_csv('Nutrition_Dataset_updated_with_labels.csv')
	list={food}
	item=dataNutrition[dataNutrition['FoodName'].isin(list)]
	print(item)
	netcarbs=float(item["Totalfat(g)"]+item["Availablecarbohydrateswithsugaralcohols(g)"]+item["Availablecarbohydrateswithoutsugaralcohol(g)"]
                   -item["Alcohol(g)"]-item["Starch(g)"]+item["Totalsaturatedfat(g)"]+item["Totalmonounsaturatedfat(g)"]+item["Totalpolyunsaturatedfat(g)"]
                   +item["Totaltransfattyacids(mg)"])
	GI=item["GyclemicIndex"].values
	GlycemicLoad=(float)(netcarbs*GI*serving)/10000.0
	print()



	if(GlycemicLoad<=10.0):
		print("Food is appropriate for consumption")
		text = 'You are totally okay to have it, Enjoy. keep monitoring your glucose level and follow the diet'
		template = '6.prediction.html'



	elif(GlycemicLoad<=19.0 and sugarlevel<=110.0):
		print("Food is appropriate for consumption")
		text = 'You are totally okay to have it, Enjoy. keep monitoring your glucose level and follow the diet'
		template = '6.prediction.html'


	elif(GlycemicLoad>=20.0 and GI<=55.0):
		print("If food is consumed in lesser quatity, it would be fit for consumption")
		text = 'If food is consumed in lesser quatity, it would be fit for consumption'
		#descision = input("Do you want a healthier food substitute(Y/N):")
		template = '5.prediction.html'

		server = smtplib.SMTP('smtp.gmail.com:587')
		server.ehlo('Gmail')
		server.starttls()
		server.login(email,password)
		fromaddr = 'From line here.'

		# The To Address
		toaddrs  = "ismaheelfaheem@gmail.com"
		server.sendmail(fromaddr,toaddrs,message.as_string())
		server.quit()



	else:
		print("Food is inappropriate for consumption")
		#descision=input("Do you want a healthier food substitute(Y/N):")
		text = "Food is inappropriate for consumption"
		template = '5.prediction.html'

		server = smtplib.SMTP('smtp.gmail.com:587')
		server.ehlo('Gmail')
		server.starttls()
		server.login(email,password)
		fromaddr = 'From line here.'

		# The To Address
		toaddrs  = "ismaheelfaheem@gmail.com"
		server.sendmail(fromaddr,toaddrs,message.as_string())
		server.quit()


	return render_template(template, text = text, food = food, serving = serving)




@app.route('/recommend',methods=['POST'])
def recommend():
	if request.method == 'POST':
		# Read teh food
		file = open("food.txt", "r")
		food = file.read()
		print(food)
		file.close()
		print(type(food))
		data = pd.read_csv('Nutrition_Dataset_updated_with_labels.csv')
		list={food}
		f=data[data['FoodName'].isin(list)]
		group = f['labels']
		others = data[data['labels'] == int(group.values)]
		GI = f['GyclemicIndex']
		data = data[data['GyclemicIndex'] < int(GI.values)]
		data = data.sort_values("GyclemicIndex")
		recom = data.iloc[[-1]]
		recom = recom['FoodName'].values
		print(recom)
		print(type(recom))
		text = 'You may have ' + recom[0] + ' in a limited quantity'


	return render_template('7.recomendation.html', text = text) # Render home.html




print(f"To acces the Gloable link please click {public_url}")
app.run(port=port_no)

To acces the Gloable link please click https://626e-35-204-148-249.ngrok-free.app
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:24:39] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:24:40] "GET /main.js HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:24:41] "GET /static/test1.png HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:24:41] "GET /static/css/style.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:24:42] "GET /main.js HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:24:42] "GET /static/bg.png HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:24:43] "GET /static/favicon.png HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:24:45] "POST /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:24:45] "GET /static/css/style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1

['User', '123', '25', 'M', '120', 'A']
lines are ['User', '123']
match


INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:25:01] "POST /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:25:02] "GET /static/css/style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [05/Apr/2024 15:25:03] "GET /static/test101.png HTTP/1.1" 200 -
